## Preprocessing Census Data

- Creating State Detail JSON for backend
- Restructuring Census Data

In [1]:
import pandas as pd
import numpy as np
import json

### Arkansas Census Data Path

In [2]:
path = '../../data/Arkansas/ar-census.csv'

### csv to pandas

In [3]:
census_df = pd.read_csv(path)

In [4]:
census_df

,Label (Grouping),Arkansas!!Estimate,Arkansas!!Margin of Error,Arkansas!!Percent,Arkansas!!Percent Margin of Error
0,SEX AND AGE,NaN,NaN,NaN,NaN
1,Total population,"3,088,354",*****,"3,088,354",(X)
2,Male,"1,517,526","±4,557",49.1%,±0.1
3,Female,"1,570,828","±4,557",50.9%,±0.1
4,Sex ratio (males per 100 females),96.6,±0.6,(X),(X)
...,...,...,...,...,...
108,Total housing units,"1,421,029",±136,(X),(X)
109,"CITIZEN, VOTING AGE POPULATION",NaN,NaN,NaN,NaN
110,"Citizen, 18 and over population","2,285,378","±6,442","2,285,378",(X)
111,Male,"1,107,768","±4,315",48.5%,±0.1


In [5]:
census_df.columns

Index(['Label (Grouping)', 'Arkansas!!Estimate', 'Arkansas!!Margin of Error',
       'Arkansas!!Percent', 'Arkansas!!Percent Margin of Error'],
      dtype='str')

### State Detail

Which is structured like the following:
```json
{
  "racialPopulation": {
    "totalPopulation": 1031890,
    "whitePopulation": 636476,
    "blackPopulation": 240431,
    "otherPopulation": 46435,
    "latinoPopulation": 108348
  },
  "voterDistribution": {
    "democratPercent": 56.5,
    "republicanPercent": 41.8,
    "otherPercent": 1.5,
    "partyControl": "Democrat"
  },
  "state": "Delaware"
}
```

#### **DEMOGRAPHIC**

#### Isolate

In [6]:
state_demo = census_df[['Label (Grouping)', 'Arkansas!!Estimate']]
state_demo

,Label (Grouping),Arkansas!!Estimate
0,SEX AND AGE,NaN
1,Total population,"3,088,354"
2,Male,"1,517,526"
3,Female,"1,570,828"
4,Sex ratio (males per 100 females),96.6
...,...,...
108,Total housing units,"1,421,029"
109,"CITIZEN, VOTING AGE POPULATION",NaN
110,"Citizen, 18 and over population","2,285,378"
111,Male,"1,107,768"


#### Cleaning

In [7]:
state_demo = state_demo.rename(columns={'Label (Grouping)' : 'Label', 'Arkansas!!Estimate' : 'Population'})
state_demo

,Label,Population
0,SEX AND AGE,NaN
1,Total population,"3,088,354"
2,Male,"1,517,526"
3,Female,"1,570,828"
4,Sex ratio (males per 100 females),96.6
...,...,...
108,Total housing units,"1,421,029"
109,"CITIZEN, VOTING AGE POPULATION",NaN
110,"Citizen, 18 and over population","2,285,378"
111,Male,"1,107,768"


In [8]:
state_demo['Label'] = state_demo['Label'].str.replace('\xa0', '').str.strip()

In [9]:
labels = ('Total population', 'White alone', 'Black or African American alone', 'American Indian and Alaska Native alone', 'Asian alone', 'Native Hawaiian and Other Pacific Islander alone', 'Some Other Race alone', 'Two or More Races', 'Hispanic or Latino (of any race)')
state_demo = state_demo.loc[state_demo['Label'].isin(labels)]
state_demo

,Label,Population
1,Total population,"3,088,354"
34,Total population,"3,088,354"
36,Two or More Races,"364,102"
76,Two or More Races,"364,102"
84,Total population,"3,088,354"
92,Total population,"3,088,354"
93,Hispanic or Latino (of any race),"294,671"
99,White alone,"2,055,864"
100,Black or African American alone,"434,109"
101,American Indian and Alaska Native alone,"13,145"


In [10]:
state_demo['Population'] = state_demo['Population'].str.replace(',', '').str.strip()
state_demo['Population'] = pd.to_numeric(state_demo['Population'], errors='coerce')
state_demo['Population'].dtype

dtype('int64')

In [11]:
state_demo = state_demo.reset_index()

In [12]:
state_demo.drop(columns='index', inplace=True)
state_demo

,Label,Population
0,Total population,3088354
1,Total population,3088354
2,Two or More Races,364102
3,Two or More Races,364102
4,Total population,3088354
5,Total population,3088354
6,Hispanic or Latino (of any race),294671
7,White alone,2055864
8,Black or African American alone,434109
9,American Indian and Alaska Native alone,13145


In [13]:
state_demo = state_demo.iloc[5:].copy()
state_demo

,Label,Population
5,Total population,3088354
6,Hispanic or Latino (of any race),294671
7,White alone,2055864
8,Black or African American alone,434109
9,American Indian and Alaska Native alone,13145
10,Asian alone,54576
11,Native Hawaiian and Other Pacific Islander alone,11636
12,Some Other Race alone,8446
13,Two or More Races,215907


In [14]:
other_labels = [
    'American Indian and Alaska Native alone',
    'Asian alone',
    'Native Hawaiian and Other Pacific Islander alone',
    'Some Other Race alone',
    'Two or More Races'
]

other_sum = state_demo.loc[state_demo['Label'].isin(other_labels), 'Population'].sum()

other_row = pd.DataFrame([{'Label' : 'Other', 'Population' : other_sum}])
state_demo = pd.concat([state_demo, other_row], ignore_index=True)
state_demo = state_demo[~state_demo['Label'].isin(other_labels)]
state_demo

,Label,Population
0,Total population,3088354
1,Hispanic or Latino (of any race),294671
2,White alone,2055864
3,Black or African American alone,434109
9,Other,303710


In [15]:
total_pop = state_demo.loc[state_demo['Label'] == 'Total population', 'Population'].iloc[0]
act = state_demo.loc[~state_demo['Label'].isin(['Total population']), 'Population'].sum()

print(f"Total population: {total_pop}")
print(f"Named total population: {act}")

Total population: 3088354
Named total population: 3088354


In [16]:
white_pop = state_demo.loc[state_demo['Label'] == 'White alone', 'Population'].iloc[0]
black_pop = state_demo.loc[state_demo['Label'] == 'Black or African American alone', 'Population'].iloc[0]
other_pop = state_demo.loc[state_demo['Label'] == 'Other', 'Population'].iloc[0]
latino_pop = state_demo.loc[state_demo['Label'] == 'Hispanic or Latino (of any race)', 'Population'].iloc[0]

print(f"White population: {white_pop}")
print(f"Black population: {black_pop}")
print(f"Other population: {other_pop}")
print(f"Latino population: {latino_pop}")

White population: 2055864
Black population: 434109
Other population: 303710
Latino population: 294671


In [17]:
racial_pop = {'totalPopulation': total_pop,  
              'whitePopulation': white_pop,
              'blackPopulation': black_pop,
              'otherPopulation': other_pop,
              'latinoPopulation': latino_pop
             }

#### **VOTER DISTRIBUTION**
Based on information from [Arkansas 2024 election data](https://enr.totalresults.com/arkansas#election=1846&filter=FED&contest=366&bucket=results) as described in GUI-2.

In [18]:
voter_dist = {'democratPercent': 33.6, 
              'republicanPercent': 64.2, 
              'otherPercent': 2.2,
              'partyControl': 'Republican'
             }

#### **MERGE AND EXPORT**

In [19]:
state_detail = {'racialPopulation': racial_pop,
                'voterDistribution': voter_dist,
                'state': 'Arkansas'
               }

In [20]:
def convert_numpy(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    
with open("../output/Arkansas/arkansas_detail.json", "w") as f:
    json.dump(state_detail, f, indent=4, default=convert_numpy)